In [2]:
from ete4 import PhyloTree
from ete4.smartview import Layout, BASIC_LAYOUT

#t = PhyloTree("../data/HumanTree/human_sequences_405_msa_gt01_fasttree.nw")  
t = PhyloTree("../data/HumanTree/human433_MFP_reorder.tree")

#midp = t.get_midpoint_outgroup()
#t.set_outgroup(midp)

root_children = t.children
if len(root_children) != 2:
    raise ValueError("Expected exactly two children under the root node")

style_class_1 = {'stroke-width': 3, 'stroke': '#7f1734'}   
style_class_2 = {'stroke-width': 3, 'stroke': '#557c99'}   

branch_styles = {}

for node in root_children[0].traverse():
    for child in node.children:
        branch_styles[child] = style_class_1

for node in root_children[1].traverse():
    for child in node.children:
        branch_styles[child] = style_class_2

def draw_node(node):
    style = branch_styles.get(node, None)
    return {'hz-line': style, 'vt-line': style} if style else {}

layout = Layout("Colored Branches for Root Subtrees", draw_node=draw_node)
t.explore(layouts=[layout, BASIC_LAYOUT])

Existing explorer available at http://127.0.0.1:5002


## Extract OR classes from the tree

The root splits into two clades, matching the two coloured subtrees above:
- `root_children[0]` &rarr; **Class I** (62 leaves)
- `root_children[1]` &rarr; **Class II** (371 leaves)

Walk the leaves of each clade and write `data/HumanTree/human433_OR_classes.csv`.

In [3]:
import csv
from pathlib import Path
from collections import Counter

# root_children[0] = Class I, root_children[1] = Class II (same split coloured above)
clade_classes = {0: "Class_I", 1: "Class_II"}

rows = []
for idx, clade in enumerate(root_children):
    or_class = clade_classes[idx]
    for leaf in clade.leaves():
        leaf_id = leaf.name                                  # e.g. "9606.P0C7T3"
        uniprot = leaf_id.split(".", 1)[1] if "." in leaf_id else leaf_id
        rows.append({"leaf_id": leaf_id, "uniprot": uniprot, "OR_class": or_class})

rows.sort(key=lambda r: (r["OR_class"], r["uniprot"]))

out_path = Path("../data/HumanTree/human433_OR_classes.csv")
with open(out_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["leaf_id", "uniprot", "OR_class"])
    writer.writeheader()
    writer.writerows(rows)

print("counts:", Counter(r["OR_class"] for r in rows))
print(f"wrote {out_path} -> {len(rows)} rows")

counts: Counter({'Class_II': 371, 'Class_I': 62})
wrote ../data/HumanTree/human433_OR_classes.csv -> 433 rows
